In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 52 samples and 13410 columns.
First few rows of the data:


,Overall survival(month),Survival state,Sample,TP53 Signture Score,TP53 Signture Group,TP53 Mutation,Gender,Age,Tumor_size,Grade,...,ACACA,TMEM191B,C11orf98,MRM1,H2AC17,H3C10,MRPL45,PAGR1,TMEM265,ADORA3
0,49.0,0,BC1,0.645576,Low,TP53_WT,Man,unknown,unknown,unknown,...,13.088213,2.154999,15.261451,2.467785,0.506301,0.019989,23.010832,18.444067,5.970439,0.846116
1,49.0,0,BC2,1.350214,High,TP53_WT,Man,Young,Small,HG,...,8.068462,1.326360,18.845911,5.124256,0.227321,0.750467,33.204845,15.243621,0.419335,1.226338
2,11.0,1,BC3,0.466355,Low,TP53_WT,Female,Old,Small,HG,...,15.422922,0.370062,19.831106,4.689168,0.359574,0.051241,28.186981,15.683324,1.816877,0.302842
3,2.0,1,BC4,0.315740,Low,TP53_WT,Man,Old,unknown,HG,...,9.595050,1.313457,30.044163,7.445334,0.418230,1.361837,37.776566,26.574846,1.288730,0.508015
4,27.0,1,BC5,0.285227,Low,TP53_WT,Female,Young,Large,HG,...,8.550369,0.903143,65.044716,10.862432,3.148769,3.156988,27.455772,52.573322,3.091669,1.063593


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for Overall survival(month) or Survival state: 0
Number of samples with complete data for both variables: 52
Number of censored samples (event_var == 0.0): 35
Number of events (event_var == 1.0): 17
Censored ratio (among complete): 35 / 52 = 0.673 (67.3%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE216037/description.json
